# CDM Agent — Quickstart

이 노트북은 `cdm-agent-client` 패키지의 기본 사용법을 보여줍니다.

## 사전 준비

1. **데몬 실행** — `cdm-agent-platform` 디렉터리에서:
   ```
   npm run dev
   ```
2. **Chrome 확장 설치** — `public/cdm-agent/chrome-extension` 폴더를 개발자 모드로 로드
3. **CDMS 페이지 열기** — 브라우저에서 대상 CDMS 페이지로 이동
4. **패키지 설치** (최초 1회):
   ```
   pip install -e "C:/Users/SunbeomGwon/CDMS-Agent"
   ```

In [ ]:
from cdm_agent_client import CDMSAgent

# 데몬 연결 (기본: http://127.0.0.1:3200)
agent = CDMSAgent(
    base_url="http://127.0.0.1:3200",
    study_id="20251216_ANZEILAX_GREEN",  # 스터디 ID (선택)
)
agent

## 1. 데몬 연결 확인

In [ ]:
print("데몬 연결 상태:", agent.ping())
print("연결된 브라우저 클라이언트:", agent.clients())

## 2. 현재 페이지 스냅샷 확인 (`inspect`)

현재 브라우저에 열린 CDMS 페이지의 상태를 가져옵니다.
- 보이는 행 목록 (`visible_rows`)
- 활성화된 버튼 (`enabled_actions`)
- 유효하지 않은 행 수 (`invalid_count`)

In [ ]:
snapshot = agent.inspect()
snapshot  # Jupyter에서 HTML 테이블로 렌더링됩니다

In [ ]:
# 텍스트로 확인
print("페이지:", snapshot.page_label)
print("URL:", snapshot.url)
print("보이는 행:", snapshot.visible_rows)
print("활성 버튼:", snapshot.enabled_actions)
print("유효하지 않은 행:", snapshot.invalid_row_labels)

## 3. 날짜 입력 (`set_date`)

페이지의 특정 행에 날짜를 입력합니다.

> `row_label`은 `inspect()` 결과의 `visible_rows`에서 확인한 값을 사용하세요.

In [ ]:
result = agent.set_date(
    row_label="Visit date",   # inspect()로 확인한 행 라벨
    value="2025-01-15",        # 입력할 날짜
)
result  # Jupyter에서 결과 카드로 렌더링됩니다

In [ ]:
print("결과:", result.outcome)   # "passed" | "failed" | "blocked"
print("성공 여부:", result.ok)

## 4. Save & Next 클릭 (`click_save_next`)

현재 페이지를 저장하고 다음 페이지로 이동합니다.

In [ ]:
nav_result = agent.click_save_next()
nav_result

## 5. 에러 처리

In [ ]:
from cdm_agent_client import StepFailedError, NoBrowserClientError, DaemonNotRunningError


try:
    result = agent.set_date("Visit date", "invalid-date")
except StepFailedError as e:
    print(f"스텝 실패: {e.outcome} — {e.reason}")
except NoBrowserClientError as e:
    print(f"브라우저 미연결: {e}")
except DaemonNotRunningError as e:
    print(f"데몬 미실행: {e}")

## 6. 복합 워크플로우 예시

페이지 상태를 확인한 뒤 날짜를 입력하고 저장하는 전형적인 패턴:

In [ ]:
# 현재 페이지 확인
snap = agent.inspect()
print(f"현재 페이지: {snap.page_label}")
print(f"입력 필요한 행 수: {snap.invalid_count}")

if snap.invalid_count > 0:
    print("유효하지 않은 행:", snap.invalid_row_labels)

# Visit date가 보이면 날짜 입력
if "Visit date" in snap.visible_rows:
    result = agent.set_date("Visit date", "2025-01-15")
    print(f"날짜 입력: {result.outcome}")

    # 저장 가능한 경우 Save & Next
    if result.ok and "Save & Next" in snap.enabled_actions:
        nav = agent.click_save_next()
        print(f"저장 결과: {nav.outcome}, 이동 후 페이지: {nav.page_after}")
else:
    print("Visit date 행이 현재 페이지에 없습니다.")